[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/07_logistic_regression/90_exercises.ipynb)

# Bài tập — Chương 07: Logistic Regression (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/07_logistic_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_classification_2d
from ml_course.utils import numerical_gradient, relative_error, repo_path, sigmoid

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Viết: hàm sigmoid và đạo hàm; binary cross-entropy; gradient; công thức odds và log-odds.

✍️ Trả lời:

**Đáp án:** $\sigma(z) = 1/(1 + e^{-z})$, $\sigma' = \sigma(1 - \sigma)$; $J = -\frac1m\sum[y\log h + (1 - y)\log(1 - h)]$; $\nabla J = \frac1m X^\top(h - y)$; odds $= p/(1 - p)$, log-odds $= \log\frac{p}{1-p} = \mathbf w^\top\mathbf x + b$.

## L1 — Giải thích bằng lời

**Câu 2.** Một bạn nói: "Logistic Regression chỉ vẽ được đường thẳng nên vô dụng với dữ liệu phức tạp". Phản biện.

✍️ Trả lời:

**Gợi ý:** ranh giới tuyến tính **theo feature**; với feature mapping (đa thức, tương tác, spline, embedding...) ranh giới trong không gian gốc có thể cong tuỳ ý. Thực tế Logistic Regression trên feature tốt (vd TF-IDF cho văn bản) là baseline rất mạnh; ngoài ra nó cho xác suất calibrated, nhanh, dễ diễn giải. Hạn chế thật: cần tự thiết kế feature.

## L2 — Tính tay

**Câu 3.** $\theta = [-4, 0.5, 1]$ (bias, $x_1$, $x_2$). Với điểm $x = (2, 2)$: tính $z$, $P(y = 1)$, lớp dự đoán. Tăng $x_2$ thêm 1 thì odds nhân với bao nhiêu?

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
z = -4 + 0.5 * 2 + 1 * 2
p = 1 / (1 + np.exp(-z))
pred_class = int(p >= 0.5)
odds_multiplier = np.exp(1.0)
### END CODE HERE ###

In [ ]:
check_close(z, -1.0, message="z")
check_close(p, 0.2689414, tol=1e-6, message="P(y=1)")
check_equal(pred_class, 0, "lớp dự đoán")
check_close(odds_multiplier, np.e, message="odds × e^θ2")

**Câu 4.** Ba mẫu với nhãn $y = [1, 0, 1]$ và xác suất dự đoán $h = [0.9, 0.2, 0.6]$. Tính binary cross-entropy (log tự nhiên).

In [ ]:
### START CODE HERE ### (≈ 1 dòng)
bce = -(np.log(0.9) + np.log(0.8) + np.log(0.6)) / 3
### END CODE HERE ###

In [ ]:
check_close(bce, 0.27978, tol=1e-4, message="BCE")

## L3 — Code

**Bài 5.** Cài đặt Logistic Regression **có L2** bằng mini-batch GD: `fit_logreg(X, y, lam, lr, epochs, batch_size, seed)` (X đã có cột bias ở đầu, **không phạt bias**). Loss: $J = \text{BCE} + \frac{\lambda}{2m}\sum_{j \ge 1}\theta_j^2$.

In [ ]:
def fit_logreg(X, y, lam=1.0, lr=0.3, epochs=200, batch_size=64, seed=0):
    r = np.random.default_rng(seed)
    m = len(y)
    theta = np.zeros(X.shape[1])
    for _ in range(epochs):
        idx = r.permutation(m)
        for s in range(0, m, batch_size):
            b = idx[s : s + batch_size]
            ### START CODE HERE ### (≈ 4 dòng)
            h = sigmoid(X[b] @ theta)
            reg = np.r_[0.0, theta[1:]] * lam / m
            grad = X[b].T @ (h - y[b]) / len(b) + reg
            theta -= lr * grad
            ### END CODE HERE ###
    return theta

In [ ]:
Xc, yc = make_classification_2d("linear", n=600, noise=0.3, seed=2)
Xc_b = np.c_[np.ones(len(Xc)), StandardScaler().fit_transform(Xc)]
th = fit_logreg(Xc_b, yc, lam=1.0)
acc = np.mean((sigmoid(Xc_b @ th) >= 0.5) == yc)
sk_acc = LogisticRegression().fit(Xc_b[:, 1:], yc).score(Xc_b[:, 1:], yc)
check_greater(acc, sk_acc - 0.02, "accuracy gần bằng sklearn")

**Bài 6.** Viết `best_threshold(y, p, cost_fp, cost_fn)` trả về ngưỡng (trong `np.linspace(0.01, 0.99, 99)`) có **tổng chi phí** nhỏ nhất.

In [ ]:
def best_threshold(y, p, cost_fp, cost_fn):
    thresholds = np.linspace(0.01, 0.99, 99)
    ### START CODE HERE ### (≈ 2 dòng)
    costs = [cost_fp * np.sum((p >= t) & (y == 0)) + cost_fn * np.sum((p < t) & (y == 1)) for t in thresholds]
    t_best = thresholds[int(np.argmin(costs))]
    ### END CODE HERE ###
    return t_best

In [ ]:
# xác suất calibrate hoàn hảo: y ~ Bernoulli(p) → ngưỡng tối ưu ≈ C_FP / (C_FP + C_FN)
p_sim = rng.uniform(0, 1, 200_000)
y_sim = (rng.random(200_000) < p_sim).astype(int)
check_close(best_threshold(y_sim, p_sim, cost_fp=1, cost_fn=9), 0.1, tol=0.03, message="ngưỡng ≈ 1/(1+9)")
check_close(best_threshold(y_sim, p_sim, cost_fp=3, cost_fn=1), 0.75, tol=0.03, message="ngưỡng ≈ 3/(3+1)")

## L4 — Debug

**Bài 7.** Hàm cost dưới đây trả về `nan` với model "tự tin" và gradient bị sai dấu nên cost **tăng** khi train. Sửa cả hai.

In [ ]:
def cost_fixed(theta, X, y):
    ### START CODE HERE ### (≈ 2 dòng)
    # STARTER: h = 1 / (1 + np.exp(-X @ theta))
    # STARTER: return -np.mean(y * np.log(h) + (1 - y) * np.log(1 - h))      # 🐛 log(0)
    h = np.clip(sigmoid(X @ theta), 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(h) + (1 - y) * np.log(1 - h))
    ### END CODE HERE ###


def grad_fixed(theta, X, y):
    ### START CODE HERE ### (≈ 1 dòng)
    # STARTER: return X.T @ (y - sigmoid(X @ theta)) / len(y)       # 🐛 dấu
    return X.T @ (sigmoid(X @ theta) - y) / len(y)
    ### END CODE HERE ###

In [ ]:
Xt = np.c_[np.ones(4), [-3.0, -1.0, 1.0, 3.0]]
yt = np.array([0, 0, 1, 1.0])
check_true(np.isfinite(cost_fixed(np.array([0.0, 500.0]), Xt, yt)), "cost hữu hạn với θ cực lớn")
tt = np.array([0.3, -0.2])
check_less(relative_error(grad_fixed(tt, Xt, yt), numerical_gradient(lambda t: cost_fixed(t, Xt, yt), tt.copy())), 1e-6, "gradient đúng dấu")

## L5 — Phân tích

**Bài 8.** Trên dữ liệu `moons` (1 000 mẫu, `noise=0.25`), so sánh ROC-AUC (5-fold CV) của: (a) Logistic Regression tuyến tính; (b) feature mapping bậc 3 với C ∈ {0.01, 1, 100}. Vẽ decision boundary cho 3 giá trị C và nhận xét bias/variance.

In [ ]:
from ml_course.visualization import plot_decision_boundary

Xm, ym = make_classification_2d("moons", n=1000, noise=0.25, seed=3)
aucs = {}
### START CODE HERE ### (≈ 5 dòng)
aucs["linear"] = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), Xm, ym, cv=5, scoring="roc_auc").mean()
for C in [0.01, 1, 100]:
    model = make_pipeline(PolynomialFeatures(3), StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    aucs[f"poly3, C={C}"] = cross_val_score(model, Xm, ym, cv=5, scoring="roc_auc").mean()
### END CODE HERE ###
print({k: round(v, 4) for k, v in aucs.items()})
fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
for ax, C in zip(axes, [0.01, 1, 100]):
    model = make_pipeline(PolynomialFeatures(3), StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(Xm, ym)
    plot_decision_boundary(model, Xm, ym, ax=ax)
    ax.set_title(f"C = {C}")
plt.show()

In [ ]:
check_true(aucs["poly3, C=1"] > aucs["linear"], "feature mapping cải thiện so với tuyến tính")

✍️ Nhận xét:

**Gợi ý:** tuyến tính underfit (ranh giới thẳng không theo được hình mặt trăng). Bậc 3 với C nhỏ (0.01) bị regularize mạnh → vẫn gần tuyến tính, bias cao; C = 1 cong vừa đủ — tốt nhất; C = 100 gần như không regularize, với bậc 3 chưa overfit nhiều vì dữ liệu nhiều (1 000 mẫu) — nhưng ranh giới đã bắt đầu "méo" ở vùng ít dữ liệu.

## L6 — Thực tế

**Bài 9.** Xây model churn với Logistic Regression trên `customer_churn.csv`: one-hot các cột phân loại, chuẩn hoá cột số, thêm feature `tenure_months × (contract == "month-to-month")`. Chọn `C` bằng `LogisticRegressionCV`. Mục tiêu **ROC-AUC test > 0.80**. In 5 odds ratio lớn nhất.

In [ ]:
df = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))
### START CODE HERE ### (≈ 10 dòng)
from sklearn.linear_model import LogisticRegressionCV

feat = pd.get_dummies(df.drop(columns=["customer_id", "churn"]), dtype=float)
feat["tenure_x_m2m"] = df["tenure_months"] * (df["contract"] == "month-to-month")
X_tr, X_te, y_tr, y_te = train_test_split(feat, df["churn"], test_size=0.25, random_state=0, stratify=df["churn"])
churn_model = make_pipeline(StandardScaler(), LogisticRegressionCV(Cs=np.logspace(-3, 2, 12), cv=5, scoring="roc_auc", max_iter=5000))
churn_model.fit(X_tr, y_tr)
test_auc = roc_auc_score(y_te, churn_model.predict_proba(X_te)[:, 1])
odds = pd.Series(np.exp(churn_model[-1].coef_.ravel()), index=feat.columns).sort_values(ascending=False)
### END CODE HERE ###
print(f"ROC-AUC test = {test_auc:.4f}")
print(odds.head(5).round(3))

In [ ]:
check_greater(test_auc, 0.80, "ROC-AUC test > 0.80")

## L7 — Nâng cao

**Bài 10.** Cài đặt **Newton's method (IRLS)** cho Logistic Regression có L2: $\theta \leftarrow \theta - H^{-1}\nabla J$, với $H = \frac1m X^\top S X + \frac{\lambda}{m}I'$, $S = \text{diag}(h(1 - h))$ ($I'$ không phạt bias). Chứng minh bằng thực nghiệm: hội tụ trong ≤ 10 bước tới cùng nghiệm với `scipy.optimize.minimize`.

In [ ]:
from scipy.optimize import minimize


def newton_logreg(X, y, lam=1.0, n_steps=10):
    m, d = X.shape
    I = np.eye(d)
    I[0, 0] = 0.0
    theta = np.zeros(d)
    for _ in range(n_steps):
        ### START CODE HERE ### (≈ 4 dòng)
        h = sigmoid(X @ theta)
        grad = X.T @ (h - y) / m + lam / m * (I @ theta)
        H = X.T @ (X * (h * (1 - h))[:, None]) / m + lam / m * I
        theta = theta - np.linalg.solve(H, grad)
        ### END CODE HERE ###
    return theta


def reg_cost(theta, X, y, lam):
    h = np.clip(sigmoid(X @ theta), 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(h) + (1 - y) * np.log(1 - h)) + lam / (2 * len(y)) * np.sum(theta[1:] ** 2)

In [ ]:
theta_newton = newton_logreg(Xc_b, yc, lam=1.0, n_steps=10)
theta_bfgs = minimize(reg_cost, np.zeros(3), args=(Xc_b, yc, 1.0), method="BFGS").x
check_close(theta_newton, theta_bfgs, tol=1e-4, message="Newton (10 bước) = BFGS")